In [ ]:
import pandas as pd
import numpy as np
import altair as alt
import seaborn as sns
import matplotlib.pyplot as plt

from scipy.stats import pearsonr, spearmanr


In [ ]:
stadium = pd.read_csv('data/processed/clean_stadium.csv')
fanbase = pd.read_csv('data/processed/fanbase_clean.csv')
merch = pd.read_csv('data/processed/cleaned_merch.csv')
fan_merch_merged = pd.read_csv('data/processed/merch_fanbase_merged.csv')

## Calculate Fan Life Time Value by Age, Region, Season Pass demographics

In [ ]:
fan_ltv = fan_merch_merged.groupby('Member_ID').agg({
    'Unit_Price': 'sum', 
    'Barcode': 'count',
    'Games_Attended': 'first',   
    'Seasonal_Pass': 'first',  
    'Customer_Age_Group': 'first',
    'Customer_Region': 'first'
}).rename(columns={
    'Unit_Price': 'Total_Merch_Spend',
    'Barcode': 'Purchase_Count'
}).reset_index()


In [ ]:

ticket_sources = ['Upper Bowl', 'Lower Bowl', 'Season', 'Premium']
total_ticket_revenue = stadium[stadium['Source'].isin(ticket_sources)]['Revenue'].sum()
total_attendance = fan_merch_merged['Games_Attended'].sum() 

avg_ticket_price = total_ticket_revenue / total_attendance

print(f"Estimated average ticket price: ${avg_ticket_price:.2f}")
print(f"Total ticket revenue in dataset: ${total_ticket_revenue:,.2f}")
print(f"Total attendance in dataset: {total_attendance:,}")

In [ ]:
fan_ltv['Ticket_Revenue'] = fan_ltv['Games_Attended'] * avg_ticket_price

fan_ltv['Merchandise_LTV'] = fan_ltv['Total_Merch_Spend']
fan_ltv['Total_LTV'] = fan_ltv['Ticket_Revenue'] + fan_ltv['Merchandise_LTV']

In [ ]:
fan_ltv

In [ ]:
# Segment by Age Group
ltv_by_age = fan_ltv.groupby('Customer_Age_Group').agg({
    'Total_LTV': ['mean', 'median', 'sum'],
    'Ticket_Revenue': 'mean',
    'Merchandise_LTV': 'mean',
    'Member_ID': 'count'
}).round(2)

# Segment by Region
ltv_by_region = fan_ltv.groupby('Customer_Region').agg({
    'Total_LTV': ['mean', 'median', 'sum'],
    'Ticket_Revenue': 'mean',
    'Merchandise_LTV': 'mean',
    'Member_ID': 'count'
}).round(2)

# Segment by Season Pass Status
ltv_by_pass = fan_ltv.groupby('Seasonal_Pass').agg({
    'Total_LTV': ['mean', 'median', 'sum'],
    'Ticket_Revenue': 'mean',
    'Merchandise_LTV': 'mean',
    'Games_Attended': 'mean',
    'Member_ID': 'count'
}).round(2)


In [ ]:
ltv_by_age

In [ ]:
ltv_by_pass

In [ ]:
ltv_by_region

We find no correlation between number of games attending and money spent on merchendise

In [ ]:
# Calculate correlation between games attended and merchandise spending
correlation_data = fan_ltv[['Games_Attended', 'Merchandise_LTV']].copy()

# Remove any rows with missing values
correlation_data = correlation_data.dropna()

# Calculate correlation coefficients
pearson_corr, pearson_p = pearsonr(correlation_data['Games_Attended'], 
                                     correlation_data['Merchandise_LTV'])
spearman_corr, spearman_p = spearmanr(correlation_data['Games_Attended'], 
                                        correlation_data['Merchandise_LTV'])

print(f"Pearson Correlation: {pearson_corr:.3f} (p-value: {pearson_p:.4f})")
print(f"Spearman Correlation: {spearman_corr:.3f} (p-value: {spearman_p:.4f})")

# Create attendance bins 
fan_ltv['Attendance_Bin'] = pd.cut(fan_ltv['Games_Attended'], 
                                     bins=[0, 3, 8, 13, 20],
                                     labels=['Low (0-3)', 'Medium (4-8)', 
                                            'High (9-13)', 'Very High (14+)'])

# Compare merchandise spending across attendance bins
merch_by_attendance = fan_ltv.groupby('Attendance_Bin').agg({
    'Merchandise_LTV': ['mean', 'median', 'sum'],
    'Member_ID': 'count',
    'Games_Attended': 'mean'
}).round(2)

merch_by_attendance


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import seaborn as sns

# Create feature set for clustering
clustering_features = fan_ltv[['Games_Attended', 'Merchandise_LTV', 'Seasonal_Pass', 'Member_ID']].copy()

# Add additional behavioral features
# Purchase frequency
purchase_freq = fan_merch_merged.groupby('Member_ID').agg({
    'Barcode': 'count',  # Number of transactions
    'Item_Category': 'nunique',  # Variety of items purchased
    'Promotion': 'mean',  # Proportion of promotional purchases
    'Channel': lambda x: (x == 'Online').sum() / len(x)  # Online purchase ratio
}).rename(columns={
    'Barcode': 'Purchase_Count',
    'Item_Category': 'Item_Variety',
    'Promotion': 'Promo_Rate',
    'Channel': 'Online_Rate'
}).reset_index()



In [ ]:
clustering_features

In [ ]:
# Merge back to clustering dataset
clustering_data = clustering_features.merge(purchase_freq, on='Member_ID', how='left')
clustering_data = clustering_data.fillna(0)  # Fill NaN for members with no purchases

# Convert boolean to numeric
clustering_data['Seasonal_Pass'] = clustering_data['Seasonal_Pass'].astype(int)

print("Clustering Features:")
print(clustering_data.describe())

In [ ]:
from sklearn.metrics import silhouette_score, davies_bouldin_score
import random

# Standardize features (important for K-means)
scaler = StandardScaler()
features_scaled = scaler.fit_transform(clustering_data)



In [ ]:
random.seed(123)

sample_size = min(10000, len(features_scaled)) 
sample_indices = np.random.choice(len(features_scaled), sample_size, replace=False)
features_sample = features_scaled[sample_indices]

# Run optimization on sample
inertias = []
silhouette_scores = []
K_range = range(2, 11)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(features_sample)
    inertias.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(features_sample, kmeans.labels_))

# Then use optimal K on full dataset
optimal_k = K_range[np.argmax(silhouette_scores)]
final_kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
final_labels = final_kmeans.fit_predict(features_scaled)

In [ ]:
# Plot results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Elbow plot
axes[0].plot(K_range, inertias, 'bo-')
axes[0].set_xlabel('Number of Clusters')
axes[0].set_ylabel('Inertia')
axes[0].set_title('Elbow Method')
axes[0].grid(True)

# Silhouette score
axes[1].plot(K_range, silhouette_scores, 'ro-')
axes[1].set_xlabel('Number of Clusters')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score by K')
axes[1].grid(True)

plt.tight_layout()
plt.show()

print("\nSilhouette Scores:")
for k, score in zip(K_range, silhouette_scores):
    print(f"K={k}: {score:.3f}")

In [ ]:

optimal_k = 4 #anything more than 4 resulted in identical clusters, despite silhouette scores and elbow

# Fit final model
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(features_scaled)

# Add cluster labels back to original data
fan_ltv['Cluster'] = cluster_labels
clustering_data['Cluster'] = cluster_labels

In [ ]:
# Calculate cluster characteristics
cluster_profiles = fan_ltv.groupby('Cluster').agg({
    'Total_LTV': ['mean', 'median', 'sum'],
    'Ticket_Revenue': 'mean',
    'Merchandise_LTV': 'mean',
    'Games_Attended': 'mean',
    'Seasonal_Pass': 'mean',  # Proportion with season pass
    'Member_ID': 'count'
}).round(2)

cluster_profiles.columns = ['_'.join(col).strip() for col in cluster_profiles.columns.values]
print("\nCLUSTER PROFILES")
print(cluster_profiles)

# Add behavioral characteristics
behavior_profiles = clustering_data.groupby('Cluster').agg({
    'Purchase_Count': 'mean',
    'Item_Variety': 'mean',
    'Promo_Rate': 'mean',
    'Online_Rate': 'mean'
}).round(2)

print("\nBEHAVIORAL PROFILES")
print(behavior_profiles)

# Demographic breakdown by cluster
demo_profiles = fan_ltv.groupby(['Cluster', 'Customer_Age_Group']).size().unstack(fill_value=0)
print("\nAGE DISTRIBUTION BY CLUSTER")
print(demo_profiles)

region_profiles = fan_ltv.groupby(['Cluster', 'Customer_Region']).size().unstack(fill_value=0)
print("\n REGION DISTRIBUTION BY CLUSTER ")
print(region_profiles)

The below code for visualizations was written with the help of Microsoft Copilot

In [ ]:
# PCA for visualization
pca = PCA(n_components=2)
features_pca = pca.fit_transform(features_scaled)

plt.figure(figsize=(12, 8))
scatter = plt.scatter(features_pca[:, 0], features_pca[:, 1], 
                     c=cluster_labels, cmap='viridis', alpha=0.6, s=50)
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
plt.title('Fan Clusters (PCA Visualization)')
plt.colorbar(scatter, label='Cluster')
plt.grid(True, alpha=0.3)
plt.show()

# Radar chart for cluster comparison
from math import pi

categories = ['Games_Attended', 'Merchandise_LTV', 'Purchase_Count', 
              'Item_Variety', 'Seasonal_Pass']

# Normalize values for radar chart (0-1 scale)
cluster_radar = clustering_data.groupby('Cluster')[categories].mean()
cluster_radar_norm = (cluster_radar - cluster_radar.min()) / (cluster_radar.max() - cluster_radar.min())

fig, axes = plt.subplots(1, optimal_k, figsize=(20, 5), subplot_kw=dict(projection='polar'))

for idx, cluster in enumerate(range(optimal_k)):
    ax = axes[idx] if optimal_k > 1 else axes
    
    values = cluster_radar_norm.iloc[cluster].values.tolist()
    values += values[:1]  # Complete the circle
    
    angles = [n / float(len(categories)) * 2 * pi for n in range(len(categories))]
    angles += angles[:1]
    
    ax.plot(angles, values, 'o-', linewidth=2)
    ax.fill(angles, values, alpha=0.25)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories, size=8)
    ax.set_ylim(0, 1)
    ax.set_title(f'Cluster {cluster}', size=12, weight='bold')
    ax.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Create interpretable cluster names based on characteristics
def name_cluster(row):
    """
    Function to assign meaningful names to clusters
    Adjust logic based on your actual cluster profiles
    """
    cluster = row['Cluster']
    games = row['Games_Attended']
    merch = row['Merchandise_LTV']
    season_pass = row['Seasonal_Pass']
    
    # Example logic - adjust based on your results
    if games > 10 and merch > 200 and season_pass == 1:
        return 'Super Fans'
    elif games > 10 and season_pass == 1:
        return 'Loyal Attendees'
    elif merch > 150:
        return 'Merchandise Enthusiasts'
    elif games < 5 and merch < 50:
        return 'Casual Fans'
    else:
        return 'Moderate Fans'


for cluster in range(optimal_k):
    cluster_data = fan_ltv[fan_ltv['Cluster'] == cluster]
    print(f"\nCluster {cluster}:")
    print(f"  Size: {len(cluster_data)} fans")
    print(f"  Avg LTV: ${cluster_data['Total_LTV'].mean():.2f}")
    print(f"  Avg Games: {cluster_data['Games_Attended'].mean():.1f}")
    print(f"  Avg Merch: ${cluster_data['Merchandise_LTV'].mean():.2f}")
    print(f"  Season Pass %: {cluster_data['Seasonal_Pass'].mean()*100:.1f}%")

Cluster 0: Merchandise Enthusiasts
- increase attendance - ticket discounts?

Cluster 1: Disengaged Fans

Cluster 2: Super Fans/VIPs
- exclusive deals and early access (important to retain)

Cluster 3: Loyal Attendees
- upsell merchendise

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# First, let's verify the cluster assignments make sense
print("="*60)
print("CLUSTER VALIDATION")
print("="*60)

cluster_summary = fan_ltv.groupby('Cluster').agg({
    'Games_Attended': ['mean', 'min', 'max'],
    'Merchandise_LTV': ['mean', 'min', 'max'],
    'Seasonal_Pass': 'mean',
    'Member_ID': 'count'
}).round(2)

print(cluster_summary)

# Check if clusters make sense
print("\n\nCluster characteristics:")
for cluster in sorted(fan_ltv['Cluster'].unique()):
    print(f"\nCluster {cluster}:")
    cluster_data = fan_ltv[fan_ltv['Cluster'] == cluster]
    print(f"  Size: {len(cluster_data)}")
    print(f"  Avg Games: {cluster_data['Games_Attended'].mean():.1f}")
    print(f"  Avg Merch: ${cluster_data['Merchandise_LTV'].mean():.2f}")
    print(f"  Season Pass %: {cluster_data['Seasonal_Pass'].mean()*100:.1f}%")

In [ ]:
# Prepare clean dataset - one row per member
print("\n" + "="*60)
print("PREPARING CLEAN DATASET")
print("="*60)

# Start fresh with fan_ltv (should already have one row per member)
model_data = fan_ltv[['Member_ID', 'Customer_Age_Group', 'Customer_Region', 
                       'Seasonal_Pass', 'Games_Attended', 'Merchandise_LTV', 
                       'Cluster']].copy()

# Remove any duplicates
model_data = model_data.drop_duplicates(subset='Member_ID')
print(f"Total members: {len(model_data)}")

# Check for missing values
print("\nMissing values:")
print(model_data.isnull().sum())

# Remove rows with missing values in key columns
model_data = model_data.dropna(subset=['Customer_Age_Group', 'Customer_Region', 'Cluster'])
print(f"After removing NaN: {len(model_data)}")

# Verify cluster distribution
print("\nCluster distribution:")
print(model_data['Cluster'].value_counts().sort_index())

In [ ]:
# One-hot encoding is often better than label encoding for tree models
print("\n" + "="*60)
print("ENCODING FEATURES")
print("="*60)

# Create a copy for modeling
modeling_df = model_data.copy()

# One-hot encode categorical variables
age_dummies = pd.get_dummies(modeling_df['Customer_Age_Group'], prefix='Age', drop_first=False)
region_dummies = pd.get_dummies(modeling_df['Customer_Region'], prefix='Region', drop_first=False)

# Combine all features
feature_df = pd.concat([
    age_dummies,
    region_dummies,
    modeling_df[['Seasonal_Pass', 'Games_Attended', 'Merchandise_LTV']].astype(float)
], axis=1)

# Convert boolean to int
if 'Seasonal_Pass' in feature_df.columns:
    feature_df['Seasonal_Pass'] = feature_df['Seasonal_Pass'].astype(int)

print(f"Feature columns: {list(feature_df.columns)}")
print(f"Feature shape: {feature_df.shape}")

# Target variable
y = modeling_df['Cluster'].values

print(f"\nTarget distribution:")
print(pd.Series(y).value_counts().sort_index())

In [ ]:
print("\n" + "="*60)
print("TRAIN-TEST SPLIT")
print("="*60)

X_train, X_test, y_train, y_test = train_test_split(
    feature_df, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y  # Ensures balanced classes
)

print(f"Training set: {len(X_train)} samples")
print(f"Test set: {len(X_test)} samples")
print(f"\nTraining cluster distribution:")
print(pd.Series(y_train).value_counts().sort_index())
print(f"\nTest cluster distribution:")
print(pd.Series(y_test).value_counts().sort_index())

In [ ]:
print("\n" + "="*60)
print("TRAINING DECISION TREE")
print("="*60)

# Train decision tree
dt_model = DecisionTreeClassifier(
    max_depth=10,
    min_samples_split=50,
    min_samples_leaf=20,
    random_state=42
)

dt_model.fit(X_train, y_train)

# Predictions
y_train_pred = dt_model.predict(X_train)
y_test_pred = dt_model.predict(X_test)

# Evaluate
train_acc = accuracy_score(y_train, y_train_pred)
test_acc = accuracy_score(y_test, y_test_pred)

print(f"Training Accuracy: {train_acc:.3f}")
print(f"Test Accuracy: {test_acc:.3f}")

print("\nTest Set Classification Report:")
print(classification_report(y_test, y_test_pred))

# Confusion Matrix
cm = confusion_matrix(y_test, y_test_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix - Decision Tree')
plt.ylabel('True Cluster')
plt.xlabel('Predicted Cluster')
plt.tight_layout()
plt.show()

In [ ]:
# Check what the model is learning
feature_importance = pd.DataFrame({
    'Feature': feature_df.columns,
    'Importance': dt_model.feature_importances_
}).sort_values('Importance', ascending=False)

print("\n" + "="*60)
print("TOP 10 MOST IMPORTANT FEATURES")
print("="*60)
print(feature_importance.head(10))

plt.figure(figsize=(10, 6))
top_features = feature_importance.head(15)
plt.barh(top_features['Feature'], top_features['Importance'])
plt.xlabel('Importance')
plt.title('Top 15 Feature Importances')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
print("\n" + "="*60)
print("TRAINING RANDOM FOREST")
print("="*60)

rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_split=30,
    min_samples_leaf=10,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

# Predictions
y_train_pred_rf = rf_model.predict(X_train)
y_test_pred_rf = rf_model.predict(X_test)

# Evaluate
train_acc_rf = accuracy_score(y_train, y_train_pred_rf)
test_acc_rf = accuracy_score(y_test, y_test_pred_rf)

print(f"Training Accuracy: {train_acc_rf:.3f}")
print(f"Test Accuracy: {test_acc_rf:.3f}")

print("\nTest Set Classification Report:")
print(classification_report(y_test, y_test_pred_rf))

# Confusion Matrix
cm_rf = confusion_matrix(y_test, y_test_pred_rf)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix - Random Forest')
plt.ylabel('True Cluster')
plt.xlabel('Predicted Cluster')
plt.tight_layout()
plt.show()

In [ ]:
# Compare models
print("\n" + "="*60)
print("MODEL COMPARISON")
print("="*60)
print(f"Decision Tree Test Accuracy: {test_acc:.3f}")
print(f"Random Forest Test Accuracy: {test_acc_rf:.3f}")

# Choose best model
if test_acc_rf > test_acc:
    best_model = rf_model
    best_model_name = "Random Forest"
    print(f"\nUsing Random Forest as final model")
else:
    best_model = dt_model
    best_model_name = "Decision Tree"
    print(f"\nUsing Decision Tree as final model")

# Save everything
joblib.dump(best_model, 'cluster_prediction_model_v2.pkl')
joblib.dump(feature_df.columns.tolist(), 'feature_columns.pkl')
print(f"\nModel saved: cluster_prediction_model_v2.pkl")
print(f"Feature columns saved: feature_columns.pkl")

In [ ]:
def predict_cluster_new(age_group, region, seasonal_pass, games_attended, merch_ltv):
    """
    Proper prediction function with one-hot encoding
    """
    try:
        # Load model and feature columns
        model = joblib.load('cluster_prediction_model_v2.pkl')
        feature_columns = joblib.load('feature_columns.pkl')
        
        # Create input dataframe
        input_data = pd.DataFrame({
            'Customer_Age_Group': [age_group],
            'Customer_Region': [region],
            'Seasonal_Pass': [int(seasonal_pass)],
            'Games_Attended': [float(games_attended)],
            'Merchandise_LTV': [float(merch_ltv)]
        })
        
        # One-hot encode
        age_dummies = pd.get_dummies(input_data['Customer_Age_Group'], prefix='Age', drop_first=False)
        region_dummies = pd.get_dummies(input_data['Customer_Region'], prefix='Region', drop_first=False)
        
        # Combine
        input_features = pd.concat([
            age_dummies,
            region_dummies,
            input_data[['Seasonal_Pass', 'Games_Attended', 'Merchandise_LTV']].astype(float)
        ], axis=1)
        
        # Ensure all expected columns exist (add missing ones as 0)
        for col in feature_columns:
            if col not in input_features.columns:
                input_features[col] = 0
        
        # Reorder columns to match training data
        input_features = input_features[feature_columns]
        
        # Predict
        cluster = model.predict(input_features)[0]
        
        # Get probabilities if available
        if hasattr(model, 'predict_proba'):
            probabilities = model.predict_proba(input_features)[0]
            prob_dict = {f"Cluster_{i}": float(prob) for i, prob in enumerate(probabilities)}
        else:
            prob_dict = {f"Cluster_{cluster}": 1.0}
        
        # Cluster names (update based on your actual clusters)
        cluster_names = {
            0: "Merchandise Enthusiasts",
            1: "Disengaged/Inactive",
            2: "Super Fans",
            3: "Loyal Attendees"
        }
        
        return {
            'predicted_cluster': int(cluster),
            'cluster_name': cluster_names.get(cluster, f"Cluster {cluster}"),
            'probabilities': prob_dict,
            'confidence': float(max(prob_dict.values()))
        }
        
    except Exception as e:
        return {"error": str(e)}

In [ ]:
print("\n" + "="*60)
print("TESTING NEW PREDICTION FUNCTION")
print("="*60)

# Get actual examples from each cluster for testing
for cluster in sorted(fan_ltv['Cluster'].unique()):
    sample = fan_ltv[fan_ltv['Cluster'] == cluster].iloc[0]
    
    print(f"\n--- Testing Cluster {cluster} Sample ---")
    print(f"Actual: {sample['Customer_Age_Group']}, {sample['Customer_Region']}, "
          f"Pass={sample['Seasonal_Pass']}, Games={sample['Games_Attended']}, "
          f"Merch=${sample['Merchandise_LTV']:.2f}")
    
    result = predict_cluster_new(
        age_group=sample['Customer_Age_Group'],
        region=sample['Customer_Region'],
        seasonal_pass=sample['Seasonal_Pass'],
        games_attended=sample['Games_Attended'],
        merch_ltv=sample['Merchandise_LTV']
    )
    
    print(f"Predicted: Cluster {result['predicted_cluster']} - {result['cluster_name']}")
    print(f"Confidence: {result['confidence']:.2%}")
    print(f"Match: {'✓' if result['predicted_cluster'] == cluster else '✗'}")

In [ ]:
# ============================================================================
# FINAL CLUSTER PREDICTION SYSTEM
# ============================================================================

import pandas as pd
import numpy as np
import joblib

class FanClusterPredictor:
    """
    Production-ready fan cluster predictor
    """
    def __init__(self, 
                 model_path='cluster_prediction_model_v2.pkl',
                 features_path='feature_columns.pkl'):
        """Initialize the predictor with saved model and features"""
        self.model = joblib.load(model_path)
        self.feature_columns = joblib.load(features_path)
        
        # Cluster definitions (update these based on your actual clusters)
        self.cluster_info = {
            0: {
                'name': 'Merchandise Enthusiasts',
                'description': 'High merchandise spending, lower attendance',
                'strategy': 'Drive stadium attendance with exclusive in-store items'
            },
            1: {
                'name': 'Disengaged/Inactive',
                'description': 'Low engagement across all metrics',
                'strategy': 'Re-engagement campaigns, special offers'
            },
            2: {
                'name': 'Super Fans',
                'description': 'High attendance and high merchandise spending',
                'strategy': 'VIP experiences, premium offerings, loyalty rewards'
            },
            3: {
                'name': 'Loyal Attendees',
                'description': 'Frequent game attendance, moderate merch spending',
                'strategy': 'Upsell merchandise at games, bundle deals'
            }
        }
    
    def predict_single(self, age_group, region, seasonal_pass, 
                      games_attended=0, merch_ltv=0):
        """
        Predict cluster for a single fan
        
        Parameters:
        -----------
        age_group : str
            Age bracket (e.g., '<18', '18-25', '26-40', '41-60', '60+')
        region : str
            Geographic region (e.g., 'Canada', 'USA', 'Japan', etc.)
        seasonal_pass : bool
            Whether fan has a season pass
        games_attended : int
            Number of games attended (default 0 for new members)
        merch_ltv : float
            Total merchandise spending (default 0 for new members)
        
        Returns:
        --------
        dict with prediction, probabilities, and recommendations
        """
        try:
            # Create input dataframe
            input_data = pd.DataFrame({
                'Customer_Age_Group': [age_group],
                'Customer_Region': [region],
                'Seasonal_Pass': [int(seasonal_pass)],
                'Games_Attended': [float(games_attended)],
                'Merchandise_LTV': [float(merch_ltv)]
            })
            
            # One-hot encode
            age_dummies = pd.get_dummies(input_data['Customer_Age_Group'], 
                                        prefix='Age', drop_first=False)
            region_dummies = pd.get_dummies(input_data['Customer_Region'], 
                                           prefix='Region', drop_first=False)
            
            # Combine features
            input_features = pd.concat([
                age_dummies,
                region_dummies,
                input_data[['Seasonal_Pass', 'Games_Attended', 'Merchandise_LTV']].astype(float)
            ], axis=1)
            
            # Ensure all expected columns exist
            for col in self.feature_columns:
                if col not in input_features.columns:
                    input_features[col] = 0
            
            # Reorder columns to match training data
            input_features = input_features[self.feature_columns]
            
            # Predict
            cluster = self.model.predict(input_features)[0]
            
            # Get probabilities
            if hasattr(self.model, 'predict_proba'):
                probabilities = self.model.predict_proba(input_features)[0]
                prob_dict = {i: float(prob) for i, prob in enumerate(probabilities)}
            else:
                prob_dict = {cluster: 1.0}
            
            # Get cluster info
            cluster_data = self.cluster_info.get(cluster, {
                'name': f'Cluster {cluster}',
                'description': 'Unknown',
                'strategy': 'Unknown'
            })
            
            return {
                'success': True,
                'cluster': int(cluster),
                'cluster_name': cluster_data['name'],
                'description': cluster_data['description'],
                'marketing_strategy': cluster_data['strategy'],
                'confidence': float(max(prob_dict.values())),
                'all_probabilities': prob_dict,
                'inputs': {
                    'age_group': age_group,
                    'region': region,
                    'seasonal_pass': seasonal_pass,
                    'games_attended': games_attended,
                    'merchandise_ltv': merch_ltv
                }
            }
            
        except Exception as e:
            return {
                'success': False,
                'error': str(e),
                'error_type': type(e).__name__
            }
    
    def predict_batch(self, members_df):
        """
        Predict clusters for multiple members
        
        Parameters:
        -----------
        members_df : DataFrame
            DataFrame with columns: Customer_Age_Group, Customer_Region,
            Seasonal_Pass, Games_Attended, Merchandise_LTV
        
        Returns:
        --------
        DataFrame with predictions added
        """
        results = []
        
        for idx, row in members_df.iterrows():
            result = self.predict_single(
                age_group=row['Customer_Age_Group'],
                region=row['Customer_Region'],
                seasonal_pass=row['Seasonal_Pass'],
                games_attended=row.get('Games_Attended', 0),
                merch_ltv=row.get('Merchandise_LTV', 0)
            )
            
            if result['success']:
                results.append({
                    'Predicted_Cluster': result['cluster'],
                    'Cluster_Name': result['cluster_name'],
                    'Confidence': result['confidence']
                })
            else:
                results.append({
                    'Predicted_Cluster': None,
                    'Cluster_Name': 'Error',
                    'Confidence': 0.0
                })
        
        results_df = pd.DataFrame(results)
        return pd.concat([members_df.reset_index(drop=True), results_df], axis=1)
    
    def get_cluster_summary(self):
        """Get summary of all clusters"""
        return pd.DataFrame([
            {
                'Cluster': k,
                'Name': v['name'],
                'Description': v['description'],
                'Strategy': v['strategy']
            }
            for k, v in self.cluster_info.items()
        ])

# ============================================================================
# Initialize predictor
# ============================================================================
predictor = FanClusterPredictor()

print("="*60)
print("FAN CLUSTER PREDICTOR - READY")
print("="*60)
print(predictor.get_cluster_summary().to_string(index=False))